In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
from imblearn.over_sampling import SMOTE
from scipy.stats.mstats import winsorize
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s4e1/train.csv')
test = pd.read_csv(r'/kaggle/input/playground-series-s4e1/test.csv')

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
train.describe()

In [ ]:
test.describe()

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train.isnull().sum()

In [ ]:
test.isnull().sum()

In [ ]:
label_encoder = LabelEncoder()
train['Geography'] =  label_encoder.fit_transform(train['Geography'])
test['Geography'] =  label_encoder.fit_transform(test['Geography'])
train['Gender'] = label_encoder.fit_transform(train['Gender'])
test['Gender'] = label_encoder.fit_transform(test['Gender'])
train = train.drop('Surname',axis=1)
test = test.drop('Surname',axis=1)

In [ ]:
train.drop(['CustomerId'], axis=1, inplace=True)
test.drop(['CustomerId'], axis=1, inplace=True)

In [ ]:
columns=['CreditScore','Balance','EstimatedSalary','Age','NumOfProducts']
for col in columns:
    train[col]=winsorize(train[col],limits=[0.05,0.1],inclusive=(True,True))
    test[col]=winsorize(test[col],limits=[0.05,0.1],inclusive=(True,True))

In [ ]:
for i in train.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(train[i])
    plt.title(i)
    plt.show()

In [ ]:
for i in test.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(test[i])
    plt.title(i)
    plt.show()

In [ ]:
X = train.drop("Exited", axis=1) 
y = train["Exited"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

In [ ]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
rf = RandomForestClassifier(n_estimators=100,random_state=42,max_depth=5,min_samples_leaf=5,min_samples_split=5,criterion='gini')
rf.fit(X_train, y_train)

In [ ]:
y_pred = rf.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
print(classification_report(y_test, y_pred))

In [ ]:
y_pred_rf = rf.predict(test)

In [ ]:
xgb = XGBClassifier(random_state=42)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
y_pred_xg=xgb.predict(test)

In [ ]:
lgb = LGBMClassifier(random_state=42)
lgb.fit(X_train, y_train)
y_pred = lgb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
y_pred_lg=lgb.predict(test)

In [ ]:
predictions = pd.DataFrame({"id": test["id"],"Exited": y_pred_lg})
predictions["Exited"] = predictions["Exited"].round(2).astype(int)

In [ ]:
predictions.to_csv("submission.csv",index=False)